# ORD Data Cleaning

Fix the BTS field labels, check data quality, and prepare the departure sample for [EDA](ord_departure_reliability_eda.ipynb).

Set `ORD_RAW_PATH` to the original extract and `ORD_DATA_DIR` to your local `cleaned_data/` folder. Defaults point to the parent directory. Run from this repository folder after `pip install -r requirements.txt`.

This notebook rebuilds cleaning in memory and checks the existing local files. It does not overwrite them.

In [1]:
from pathlib import Path
import hashlib
import json
import os
import pandas as pd
from IPython.display import display
import numpy as np

raw_path = Path(os.environ.get('ORD_RAW_PATH', '../ord_departures_2023_2026.parquet'))
data_dir = Path(os.environ.get('ORD_DATA_DIR', '../cleaned_data'))
audit = json.loads((data_dir / 'cleaning_summary.json').read_text())
source_hash = hashlib.file_digest(raw_path.open('rb'), 'sha256').hexdigest()
assert source_hash == audit['input_sha256']
raw = pd.read_parquet(raw_path)
print(f'{len(raw):,} raw rows × {raw.shape[1]} source columns; source SHA-256 verified')
MASTER_FILE = "ord_departures_2023_2026_cleaned.parquet"
READY_FILE = "ord_departures_2023_2026_analysis_ready.parquet"
REQUIRED = ['FLIGHT_DATE', 'MARKETING_CARRIER', 'MARKETING_FLIGHT_NUMBER', 'ORIGIN', 'DEST', 'CRS_DEP_TIME', 'SOURCE_ROW_ID', 'SOURCE_MONTH', 'OPERATING_CARRIER', 'DUPLICATE_MARKER', 'CANCELLATION_CODE', 'ELIGIBLE_DEPARTURE_ANALYSIS', 'DEP_DELAY_15', 'POSITIVE_DEP_DELAY_MINUTES', 'DEP_DELAY', 'ARR_DELAY', 'TAXI_OUT', 'AIR_TIME', 'ACTUAL_ELAPSED_TIME', 'DEP_TIME', 'SCHEDULED_DEP_HOUR', 'CANCELLED', 'DIVERTED', 'DIV_AIRPORT_LANDINGS', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY', 'FIELD_031', 'FIELD_032', 'FIELD_033', 'FIELD_034', 'FIELD_035', 'FIELD_012', 'FIELD_015', 'CRS_ARR_TIME', 'SOURCE_CANCELLED_FIELD', 'SOURCE_DIVERTED_FIELD', 'OAG_CRS_DEP_DIFF', 'OAG_CRS_ARR_DIFF']


1,131,245 raw rows × 72 source columns; source SHA-256 verified


## 1. Check the source fields

[BTS field definitions](https://www.transtats.bts.gov/Fields.asp?gnoyr_VQ=FGK) and the [Form 1 layout](https://esubmit.rita.dot.gov/On-Time-Form1.aspx) explain the fixes:

- `FIELD_031`–`FIELD_035`: carrier, weather, NAS, security, and late-aircraft arrival-delay minutes.
- Original `CANCELLED` / `DIVERTED`: schedule differences, preserved under separate names.
- `FIELD_012` / `FIELD_015`: corrected CRS departure / arrival clocks; original clocks are retained as OAG times.
- Landing codes 1–5 indicate diversion; code 9 or a cancellation code indicates cancellation.

In [2]:
source_schema = pd.DataFrame({'dtype': raw.dtypes.astype(str), 'missing_rows': raw.isna().sum(), 'unique_values': raw.nunique()})
display(source_schema)
display(pd.read_csv(data_dir / 'delay_cause_mapping.csv'))

,dtype,missing_rows,unique_values
SOURCE_MONTH,object,0,41
MARKETING_CARRIER,object,0,8
MARKETING_FLIGHT_NUMBER,object,0,5891
FIELD_003,object,1131019,6
FIELD_004,object,1131019,156
...,...,...,...
FIELD_067,object,0,1
FIELD_068,object,0,1
FIELD_069,object,1131245,0
DATA_SOURCE,object,0,3


,source_column,cleaned_column,units,basis
0,FIELD_031,CARRIER_DELAY,minutes,"Form 1 fields X, Y, Z, AA, AB in record order;..."
1,FIELD_032,WEATHER_DELAY,minutes,"Form 1 fields X, Y, Z, AA, AB in record order;..."
2,FIELD_033,NAS_DELAY,minutes,"Form 1 fields X, Y, Z, AA, AB in record order;..."
3,FIELD_034,SECURITY_DELAY,minutes,"Form 1 fields X, Y, Z, AA, AB in record order;..."
4,FIELD_035,LATE_AIRCRAFT_DELAY,minutes,"Form 1 fields X, Y, Z, AA, AB in record order;..."


## 2. Clean values and fix the schema

Remove exact duplicates, normalize missing text, and keep source values alongside the corrected fields. Early delays stay negative. Clocks accept both 0000 and 2400; negative physical durations become missing.

In [3]:
exact_duplicates = int(raw.duplicated().sum())
duplicate_mask = raw.duplicated(keep='first')
clean = raw.loc[~duplicate_mask].copy()
clean.insert(0, 'SOURCE_ROW_ID', clean.index.to_numpy())
clean.reset_index(drop=True, inplace=True)
missing_tokens = ['', 'NA', 'N/A', 'NULL', 'NONE', 'NAN']
for col in raw.columns:
    if pd.api.types.is_string_dtype(clean[col]) or clean[col].dtype == object:
        text = clean[col].astype('string').str.strip()
        clean[col] = text.mask(text.str.upper().isin(missing_tokens))
print(f'{len(clean):,} rows retained; {exact_duplicates:,} exact duplicates removed.')

# Reconstruct the supported portion of the Form 1 schema, retaining source columns.
clean['SOURCE_CANCELLED_FIELD'] = clean['CANCELLED']
clean['SOURCE_DIVERTED_FIELD'] = clean['DIVERTED']
clean['OAG_CRS_DEP_DIFF'] = clean['CANCELLED']
clean['OAG_CRS_ARR_DIFF'] = clean['DIVERTED']
clean['OAG_DEP_TIME'] = clean['CRS_DEP_TIME']
clean['OAG_ARR_TIME'] = clean['CRS_ARR_TIME']
clean['CRS_DEP_TIME'] = clean['FIELD_012']
clean['CRS_ARR_TIME'] = clean['FIELD_015']
cause_mapping = {'FIELD_031': 'CARRIER_DELAY', 'FIELD_032': 'WEATHER_DELAY',
    'FIELD_033': 'NAS_DELAY', 'FIELD_034': 'SECURITY_DELAY', 'FIELD_035': 'LATE_AIRCRAFT_DELAY'}
for original, name in cause_mapping.items():
    clean[name] = clean[original]
cause_columns = list(cause_mapping.values())
clean['DIV_AIRPORT_LANDINGS'] = pd.to_numeric(clean.FIELD_039, errors='coerce').astype('Float64')
landings = clean.DIV_AIRPORT_LANDINGS
clean['CANCELLED'] = (clean.CANCELLATION_CODE.notna() | landings.eq(9).fillna(False)).astype('Int64')
clean['DIVERTED'] = landings.between(1, 5).where(landings.isin([0, 1, 2, 3, 4, 5, 9])).astype('Int64')
clean['STATUS_DERIVATION'] = 'Cancellation code/landing code 9; diversion landing codes 1–5'

1,131,245 rows retained; 0 exact duplicates removed.


### Parse dates, minutes, and clocks

In [4]:
quality = pd.DataFrame(index=clean.index)
clean['FLIGHT_DATE'] = pd.to_datetime(clean['FL_DATE_YYYYMMDD'], format='%Y%m%d', errors='coerce').astype('datetime64[us]')
quality['INVALID_FLIGHT_DATE'] = clean.FLIGHT_DATE.isna()
quality['NON_ORD_ORIGIN'] = clean.ORIGIN.ne('ORD').fillna(True)
source_month = pd.to_datetime(clean.SOURCE_MONTH, format='%Y%m', errors='coerce')
quality['SOURCE_MONTH_MISMATCH'] = (source_month.isna() | clean.FLIGHT_DATE.isna() |
    source_month.dt.to_period('M').ne(clean.FLIGHT_DATE.dt.to_period('M')))

numeric = ['CANCELLED', 'DIVERTED', 'DAY_OF_WEEK', 'DEP_DELAY', 'ARR_DELAY',
    'CRS_ELAPSED_TIME', 'ACTUAL_ELAPSED_TIME', 'ELAPSED_TIME_DIFF',
    'TAXI_OUT', 'TAXI_IN', 'AIR_TIME']
clock = ['CRS_DEP_TIME', 'DEP_TIME', 'CRS_ARR_TIME', 'ARR_TIME', 'WHEELS_OFF', 'WHEELS_ON']
conversion_records = []
numeric += ['OAG_CRS_DEP_DIFF', 'OAG_CRS_ARR_DIFF', 'DIV_AIRPORT_LANDINGS'] + cause_columns
clock += ['OAG_DEP_TIME', 'OAG_ARR_TIME']
for col in numeric + clock:
    clean['RAW_' + col] = clean[col]
    converted = pd.to_numeric(clean[col], errors='coerce').astype('Float64')
    invalid = clean[col].notna() & (converted.isna() | ~np.isfinite(converted.fillna(0)))
    quality['INVALID_NUMERIC_' + col] = invalid
    clean[col] = converted.mask(invalid)
    conversion_records.append({'column': col, 'invalid_numeric_rows': int(invalid.sum())})

for col in ['CRS_ELAPSED_TIME', 'ACTUAL_ELAPSED_TIME', 'TAXI_OUT', 'TAXI_IN', 'AIR_TIME']:
    negative = clean[col].lt(0).fillna(False)
    quality['NEGATIVE_' + col] = negative
    clean.loc[negative, col] = pd.NA

for col in ['CANCELLED', 'DIVERTED']:
    invalid = ~clean[col].isin([0, 1])
    quality['INVALID_STATUS_' + col] = invalid
    clean[col] = clean[col].where(~invalid).astype('Int64')

quality['INVALID_DAY_OF_WEEK'] = ~clean.DAY_OF_WEEK.isin(range(1, 8))
clean['DAY_OF_WEEK'] = clean.DAY_OF_WEEK.where(~quality.INVALID_DAY_OF_WEEK).astype('Int64')
quality['WEEKDAY_MISMATCH'] = (clean.DAY_OF_WEEK.ne(clean.FLIGHT_DATE.dt.dayofweek + 1).fillna(True))
clean['CALENDAR_DAY_OF_WEEK'] = (clean.FLIGHT_DATE.dt.dayofweek + 1).astype('Int64')

def valid_hhmm(s):
    return (s.notna() & s.eq(s.round()) &
            (((s >= 0) & (s < 2400) & ((s % 100) < 60)) | s.eq(2400))).fillna(False)

for col in clock:
    valid = valid_hhmm(clean[col])
    quality['INVALID_CLOCK_' + col] = ~valid
    clean[col] = clean[col].where(valid).astype('Int64')
    clean[col + '_MINUTE_OF_DAY'] = (((clean[col] // 100) * 60 + clean[col] % 100) % 1440).astype('Int64')
    clean[col + '_NEXT_DAY'] = clean[col].eq(2400).astype('boolean')
clean['SCHEDULED_DEP_HOUR'] = (clean.CRS_DEP_TIME_MINUTE_OF_DAY // 60).astype('Int64')
clean['YEAR'] = clean.FLIGHT_DATE.dt.year.astype('Int64')
clean['MONTH'] = clean.FLIGHT_DATE.dt.month.astype('Int64')


# Delay causes must be nonnegative; retain numbered source fields for auditing.
for col in cause_columns:
    negative = clean[col].lt(0).fillna(False)
    quality['NEGATIVE_' + col] = negative
    clean.loc[negative, col] = pd.NA
clean['DELAY_CAUSE_TOTAL'] = clean[cause_columns].sum(axis=1, min_count=len(cause_columns))
quality['CAUSE_SUM_MISMATCH'] = (clean.DELAY_CAUSE_TOTAL.gt(0) &
    clean.DELAY_CAUSE_TOTAL.ne(clean.ARR_DELAY)).fillna(True)
quality['LATE_ARRIVAL_WITHOUT_CAUSES'] = (clean.ARR_DELAY.ge(15) &
    clean.DELAY_CAUSE_TOTAL.eq(0)).fillna(False)
# Validate the positional mapping; sums do not by themselves identify each cause.
reported = clean.DELAY_CAUSE_TOTAL.gt(0).fillna(False)
assert reported.any()
assert clean.loc[reported, 'DELAY_CAUSE_TOTAL'].eq(clean.loc[reported, 'ARR_DELAY']).all()
print(f'Validated cause sums on {int(reported.sum()):,} nonzero-cause rows.')
# Report schedule-difference checks, allowing for unvalidated day-boundary conventions.
for oag, crs, difference in [('OAG_DEP_TIME', 'CRS_DEP_TIME', 'OAG_CRS_DEP_DIFF'),
                             ('OAG_ARR_TIME', 'CRS_ARR_TIME', 'OAG_CRS_ARR_DIFF')]:
    oag_min = clean[oag] // 100 * 60 + clean[oag] % 100
    crs_min = clean[crs] // 100 * 60 + clean[crs] % 100
    matches = (oag_min - crs_min).eq(clean[difference]).fillna(False)
    quality['SCHEDULE_DIFFERENCE_MISMATCH_' + difference] = ~matches
    print(difference, f'{matches.mean():.6%} match OAG minus CRS in minutes')

Validated cause sums on 276,562 nonzero-cause rows.
OAG_CRS_DEP_DIFF 99.999293% match OAG minus CRS in minutes
OAG_CRS_ARR_DIFF 99.999293% match OAG minus CRS in minutes


### Mark quality issues

In [5]:
quality['HAS_CANCELLATION_CODE'] = clean.CANCELLATION_CODE.notna()
quality['CANCELLATION_STATUS_CONFLICT'] = quality.HAS_CANCELLATION_CODE & clean.CANCELLED.eq(0).fillna(False)
quality['SOURCE_DUPLICATE_MARKER'] = clean.DUPLICATE_MARKER.eq('Y').fillna(False)
quality['UNKNOWN_DUPLICATE_MARKER'] = ~clean.DUPLICATE_MARKER.isin(['N', 'Y'])
quality['NONPOSITIVE_AIR_TIME'] = clean.AIR_TIME.le(0).fillna(True)
quality['NONPOSITIVE_ACTUAL_ELAPSED_TIME'] = clean.ACTUAL_ELAPSED_TIME.le(0).fillna(True)
key = ['FLIGHT_DATE', 'MARKETING_CARRIER', 'MARKETING_FLIGHT_NUMBER', 'ORIGIN', 'DEST', 'CRS_DEP_TIME']
quality['MISSING_FLIGHT_KEY'] = clean[key].isna().any(axis=1)
quality['REPEATED_FLIGHT_KEY'] = clean.duplicated(key, keep=False)
quality = quality.fillna(False).astype(bool)
for col in quality:
    clean['QUALITY_' + col] = quality[col]
quality_report = quality.sum().sort_values(ascending=False).rename('flagged_rows').to_frame()
quality_report['flagged_pct'] = quality_report.flagged_rows / len(clean) * 100


### Select departures for analysis

In [6]:
exclusion_rules = pd.DataFrame({
    'invalid_date': quality.INVALID_FLIGHT_DATE,
    'non_ord_origin': quality.NON_ORD_ORIGIN,
    'source_month_mismatch': quality.SOURCE_MONTH_MISMATCH,
    'missing_key': quality.MISSING_FLIGHT_KEY,
    'repeated_key': quality.REPEATED_FLIGHT_KEY,
    'source_duplicate_marker': quality.SOURCE_DUPLICATE_MARKER,
    'unknown_duplicate_marker': quality.UNKNOWN_DUPLICATE_MARKER,
    'not_verified_non_cancelled': ~clean.CANCELLED.eq(0).fillna(False),
    'not_verified_non_diverted': ~clean.DIVERTED.eq(0).fillna(False),
    'cancellation_code_present': quality.HAS_CANCELLATION_CODE,
    'invalid_scheduled_departure_clock': quality.INVALID_CLOCK_CRS_DEP_TIME,
    'invalid_actual_departure_clock': quality.INVALID_CLOCK_DEP_TIME,
    'missing_departure_delay': clean.DEP_DELAY.isna(),
    'no_positive_air_time': quality.NONPOSITIVE_AIR_TIME,
    'no_positive_actual_elapsed': quality.NONPOSITIVE_ACTUAL_ELAPSED_TIME,
    'missing_or_invalid_taxi_out': clean.TAXI_OUT.isna(),
}).astype(bool)
clean['ELIGIBLE_DEPARTURE_ANALYSIS'] = ~exclusion_rules.any(axis=1)
reasons = pd.Series('', index=clean.index, dtype='string')
for rule in exclusion_rules:
    reasons += pd.Series(np.where(exclusion_rules[rule], rule + '|', ''), index=clean.index, dtype='string')
clean['EXCLUSION_REASONS'] = reasons.str.rstrip('|')
clean['DEP_DELAY_15'] = clean.DEP_DELAY.ge(15).astype('boolean')
clean['POSITIVE_DEP_DELAY_MINUTES'] = clean.DEP_DELAY.clip(lower=0)
master = clean
contract = {
    'rows': len(master),
    'analysis_ready_rows': int(master.ELIGIBLE_DEPARTURE_ANALYSIS.sum()),
    'excluded_rows': int((~master.ELIGIBLE_DEPARTURE_ANALYSIS).sum()),
    'cause_nonzero_rows': int(reported.sum()),
    'cause_sum_mismatches': int(quality.CAUSE_SUM_MISMATCH.sum()),
}
display(pd.Series(contract, name='recomputed audit').to_frame())


,recomputed audit
rows,1131245
analysis_ready_rows,1109123
excluded_rows,22122
cause_nonzero_rows,276562
cause_sum_mismatches,0


## 3. Review quality flags

Flight keys use date, marketing network, flight number, origin, destination, and corrected CRS departure time. All repeated keys are set aside. Rule counts overlap, so they are not added together.

In [7]:
rules = exclusion_rules
rule_counts = rules.sum().rename_axis('rule').rename('rows').reset_index()
expected_rules = pd.read_csv(data_dir / 'exclusion_rules.csv')
pd.testing.assert_frame_equal(rule_counts, expected_rules)
display(rule_counts.sort_values('rows', ascending=False))
quality = master.filter(regex='^QUALITY_')
quality_counts = quality.sum().rename('flagged_rows').to_frame()
quality_counts.index = quality_counts.index.str.removeprefix('QUALITY_')
quality_counts['flagged_pct'] = quality_counts.flagged_rows / len(master)*100
frozen_quality = pd.read_csv(data_dir / 'quality_flags.csv').set_index('flag')
pd.testing.assert_frame_equal(quality_counts.sort_index().rename_axis('flag'), frozen_quality.sort_index(), check_dtype=False)
display(quality_counts.sort_values('flagged_rows', ascending=False).head(20))

,rule,rows
13,no_positive_air_time,22087
14,no_positive_actual_elapsed,19437
7,not_verified_non_cancelled,19039
9,cancellation_code_present,19038
8,not_verified_non_diverted,3049
5,source_duplicate_marker,96
4,repeated_key,68
0,invalid_date,0
1,non_ord_origin,0
2,source_month_mismatch,0


,flagged_rows,flagged_pct
NONPOSITIVE_AIR_TIME,22087,1.952451
NONPOSITIVE_ACTUAL_ELAPSED_TIME,19437,1.718195
HAS_CANCELLATION_CODE,19038,1.682925
LATE_ARRIVAL_WITHOUT_CAUSES,2650,0.234255
SOURCE_DUPLICATE_MARKER,96,0.008486
REPEATED_FLIGHT_KEY,68,0.006011
SCHEDULE_DIFFERENCE_MISMATCH_OAG_CRS_DEP_DIFF,8,0.000707
SCHEDULE_DIFFERENCE_MISMATCH_OAG_CRS_ARR_DIFF,8,0.000707
INVALID_STATUS_DIVERTED,0,0.000000
INVALID_CLOCK_WHEELS_ON,0,0.000000


## 4. Check the sample size

Keep departures with valid dates, clocks, flight keys, statuses, delays, and physical durations. No delays are imputed and no outliers are trimmed.

In [8]:
ready_mask = master.ELIGIBLE_DEPARTURE_ANALYSIS
assert int(ready_mask.sum()) == audit['analysis_ready_rows'] == 1109123
assert int((~ready_mask).sum()) == audit['excluded_rows'] == 22122
assert len(master) == audit['raw_rows'] == 1131245
assert contract['cause_nonzero_rows'] == audit['cause_nonzero_rows'] == 276562
assert contract['cause_sum_mismatches'] == audit['cause_sum_mismatches'] == 0
assert str(master.FLIGHT_DATE.min().date()) == audit['first_date']
assert str(master.FLIGHT_DATE.max().date()) == audit['last_date']
display(pd.DataFrame({'population':['Raw extract','Analysis-ready','Excluded (union)'], 'rows':[len(raw),int(ready_mask.sum()),int((~ready_mask).sum())]}))

,population,rows
0,Raw extract,1131245
1,Analysis-ready,1109123
2,Excluded (union),22122


## 5. Compare with the local cleaned files

Check the rebuilt data against the existing master and analysis-ready files, then compare schema and audit counts. Confirm that the original input hash is unchanged.

In [9]:
local_master = pd.read_parquet(data_dir / MASTER_FILE, columns=list(REQUIRED))
pd.testing.assert_frame_equal(master[list(REQUIRED)].reset_index(drop=True), local_master, check_exact=True)
del local_master
local_ready = pd.read_parquet(data_dir / READY_FILE, columns=list(REQUIRED))
pd.testing.assert_frame_equal(master.loc[ready_mask,list(REQUIRED)].reset_index(drop=True), local_ready, check_exact=True)
assert local_ready.ELIGIBLE_DEPARTURE_ANALYSIS.all()
del local_ready
schema_after = pd.DataFrame({'cleaned_dtype': master.dtypes.astype(str), 'missing_after': master.isna().sum()})
frozen_schema = pd.read_csv(data_dir / 'schema_audit.csv').set_index('column')
# pandas 2 stores scalar text as object; the source audit used pandas 3's str.
# Normalize this one descriptive label only after checking every value.
assert master['STATUS_DERIVATION'].eq('Cancellation code/landing code 9; diversion landing codes 1–5').all()
if schema_after.loc['STATUS_DERIVATION','cleaned_dtype'] == 'object':
    assert frozen_schema.loc['STATUS_DERIVATION','cleaned_dtype'] == 'str'
    schema_after.loc['STATUS_DERIVATION','cleaned_dtype'] = 'str'
pd.testing.assert_frame_equal(schema_after.sort_index().rename_axis('column'), frozen_schema[['cleaned_dtype','missing_after']].sort_index())
assert hashlib.file_digest(raw_path.open('rb'), 'sha256').hexdigest() == source_hash
print('Verified source hash, schema audit, quality flags, exclusion counts, and both local cleaned files.')
print('Cause sums: 0 mismatches across 276,562 nonzero-cause rows in the cleaned master.')
del raw, master, clean, quality

Verified source hash, schema audit, quality flags, exclusion counts, and both local cleaned files.
Cause sums: 0 mismatches across 276,562 nonzero-cause rows in the cleaned master.


## Next: explore departure delays

[Open the EDA notebook](ord_departure_reliability_eda.ipynb). Some numbered fields and schedule differences still need investigation; the analysis uses the mapped fields above.